# Comparative Study of ML and DL Classification Models for Software Defect Detection and Localization

**Full experimental pipeline — Chapter 3 Methodology Implementation**

Datasets:
1. **PROMISE (NASA MDP)** — KC1, PC1, PC2 via `kagglehub.dataset_download("radowanulhaque/software-defect")` — Pipeline A (ML) only, no source code available.
2. **Defects4J** — 6 core projects (Chart, Closure, Lang, Math, Mockito, Time) — harmonised to method-level metric features for **Pipeline A**, and used as raw Java source for **Pipeline B** (DL).

Models: SVM, Decision Tree, Random Forest (Pipeline A) · CNN, LSTM, hybrid CNN-LSTM (Pipeline B)

**Run environment:** this notebook is written for a GPU-enabled Colab / Kaggle Notebook session (Section 3.9). Sections that need network access (kagglehub download, `git clone` of Defects4J) or a GPU will not execute inside a network-isolated sandbox — run this notebook end-to-end in Colab/Kaggle.

**Quick-test vs full run:** set `QUICK_TEST = True` in Cell 2 for a fast, low-resource smoke-test of the whole pipeline (small bug sample, 3 folds, 2 epochs). Set `QUICK_TEST = False` for the full protocol described in Chapter 3 (10-fold CV, all bugs, full epoch budget) — this will take substantially longer and should be run on a GPU runtime.

## 1. Setup and configuration (Section 3.9)

In [ ]:
# If running fresh on Colab/Kaggle, uncomment:
# !pip install -q kagglehub imbalanced-learn javalang statsmodels tensorflow

import os, re, time, json, random, warnings, subprocess, difflib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")

# ---- Master run-mode switch -------------------------------------------------
QUICK_TEST = True   # True = fast smoke test; False = full Chapter-3 protocol

if QUICK_TEST:
    N_OUTER_ML       = 3          # Section 3.6 specifies k=10 for the full run
    N_OUTER_DL       = 3
    BUGS_PER_PROJECT = 5          # cap Defects4J bugs per project (full run: all bugs, Table 3.1)
    DL_EPOCHS        = 3
    DL_SEARCH_ITERS  = 2
else:
    N_OUTER_ML       = 10
    N_OUTER_DL       = 10
    BUGS_PER_PROJECT = None       # None = all bugs
    DL_EPOCHS        = 30
    DL_SEARCH_ITERS  = 10

PC2_REPEATS = 5   # Section 3.3.4/3.6: PC2 uses repeated stratified 5-fold, not 10-fold

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("QUICK_TEST =", QUICK_TEST)

## 2. PROMISE (NASA MDP) dataset acquisition (Section 3.3.2, Increment I1)

Downloads the Kaggle-hosted PROMISE/NASA defect data and auto-detects individual
project files (KC1 / PC1 / PC2, or a generic train/test split) rather than assuming
a fixed filename, since Kaggle mirrors of this repository vary in structure.

In [ ]:
import kagglehub

kaggle_path = kagglehub.dataset_download("radowanulhaque/software-defect")
print("Path to dataset files:", kaggle_path)

data_files = []
for root, _, files in os.walk(kaggle_path):
    for f in files:
        if f.lower().endswith((".csv", ".arff")):
            data_files.append(os.path.join(root, f))
print(f"{len(data_files)} data file(s) found:")
for f in data_files:
    print(" -", f)

In [ ]:
def load_arff_or_csv(path):
    """Loads a .csv or a PROMISE-style .arff file into a DataFrame."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    # minimal ARFF reader: PROMISE files are simple @attribute / @data blocks
    cols, rows, in_data = [], [], False
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("%"):
                continue
            low = line.lower()
            if low.startswith("@attribute"):
                cols.append(line.split()[1].strip("'\""))
            elif low.startswith("@data"):
                in_data = True
            elif in_data:
                rows.append([v.strip() for v in line.split(",")])
    return pd.DataFrame(rows, columns=cols)


def name_dataset(filename):
    """Best-effort mapping of a filename to KC1 / PC1 / PC2 / other."""
    base = os.path.basename(filename).lower()
    for key in ("kc1", "pc1", "pc2"):
        if key in base:
            return key.upper()
    return os.path.splitext(os.path.basename(filename))[0]


promise_raw = {}
for f in data_files:
    try:
        df = load_arff_or_csv(f)
        name = name_dataset(f)
        # if multiple files map to the same name (e.g. train/test), keep both parts
        if name in promise_raw:
            df = pd.concat([promise_raw[name], df], ignore_index=True)
        promise_raw[name] = df
    except Exception as e:
        print(f"skipped {f}: {e}")

for name, df in promise_raw.items():
    print(name, df.shape)

In [ ]:
def find_label_col(df):
    candidates = [c for c in df.columns if c.lower() in
                  ("defective", "defects", "defect", "target", "label", "class", "problems")]
    return candidates[0] if candidates else df.columns[-1]

def to_binary(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_numeric_dtype(series):
        return series.astype(float).astype(int)
    pos_vals = {"y", "yes", "true", "1", "defective", "d", "true.", "b'yes'"}
    return series.astype(str).str.strip().str.lower().str.strip("'").isin(pos_vals).astype(int)

promise_label_cols = {name: find_label_col(df) for name, df in promise_raw.items()}

for name, df in promise_raw.items():
    col = promise_label_cols[name]
    y = to_binary(df[col])
    print(f"{name}: n={len(df)}, label_col='{col}', defective={y.sum()}, "
          f"rate={y.mean()*100:.2f}%, imbalance=1:{(len(y)-y.sum())/max(y.sum(),1):.1f}")

## 3. Defects4J acquisition (Section 3.3.1, Increment I1)

Clones and initialises Defects4J and restricts to the six original-release projects
used in the proposal (Chart, Closure, Lang, Math, Mockito, Time — Table 3.1). This cell
needs the Java 11 / Git / Subversion / Perl toolchain described in Section 3.9 and
network access — run it on Colab/Kaggle, not in a network-isolated sandbox.

In [ ]:
D4J_HOME = os.path.abspath("defects4j")
D4J_PROJECTS = ["Chart", "Closure", "Lang", "Math", "Mockito", "Time"]  # Table 3.1

if not os.path.exists(D4J_HOME):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rjust/defects4j", D4J_HOME], check=True)

# System deps (Colab/Kaggle Linux runtime) -- uncomment on first run:
# !apt-get update -qq && apt-get install -y -qq subversion perl cpanminus openjdk-11-jdk
# !cd {D4J_HOME} && cpanm --installdeps . --notest && ./init.sh

os.environ["PATH"] = os.environ["PATH"] + os.pathsep + os.path.join(D4J_HOME, "framework", "bin")

def d4j(*args, cwd=None):
    result = subprocess.run(["defects4j", *args], capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout

print(d4j("info", "-p", "Lang")[:500])

In [ ]:
import io

def bug_metadata(project):
    csv_text = d4j("query", "-p", project, "-q",
                    "classes.modified,revision.id.buggy,revision.id.fixed")
    df = pd.read_csv(io.StringIO(csv_text))
    df["project"] = project
    return df

d4j_meta = pd.concat([bug_metadata(p) for p in D4J_PROJECTS], ignore_index=True)
if BUGS_PER_PROJECT is not None:
    d4j_meta = d4j_meta.groupby("project", group_keys=False).head(BUGS_PER_PROJECT)

print(d4j_meta.shape)
d4j_meta.head()

## 4. Distribution analysis of both benchmarks (Section 3.3.4, Figure 3.1)

Reproduces the four panels described in Section 3.3.4: bugs per Defects4J project,
class composition of the PROMISE datasets, Defects4J defect density, and positive-class
rate across all sources on a common axis.

In [ ]:
d4j_bug_counts = d4j_meta.groupby("project").size()

fig, ax = plt.subplots(2, 2, figsize=(12, 9))

# (a) bugs per Defects4J project
d4j_bug_counts.plot(kind="bar", ax=ax[0, 0], color="#4A6FA5")
ax[0, 0].set_title("(a) Reproducible bugs per Defects4J project")
ax[0, 0].set_ylabel("Bug count")

# (b) class composition of the PROMISE datasets
promise_counts = pd.DataFrame({
    name: to_binary(df[promise_label_cols[name]]).value_counts().rename({0: "Non-defective", 1: "Defective"})
    for name, df in promise_raw.items()
}).T
promise_counts.plot(kind="bar", stacked=True, ax=ax[0, 1], color=["#4A6FA5", "#C1443C"])
ax[0, 1].set_title("(b) Class composition of the PROMISE datasets")

# (c) Defects4J defect density (bugs per KLOC) -- Table 3.1 reference values
KLOC_TABLE = {"Chart": 96, "Closure": 90, "Lang": 22, "Math": 85, "Mockito": 23, "Time": 28}
density = (d4j_bug_counts / pd.Series(KLOC_TABLE)).dropna()
density.plot(kind="bar", ax=ax[1, 0], color="#7A5195")
ax[1, 0].axhline(density.mean(), color="black", linestyle="--", linewidth=1, label="pooled mean")
ax[1, 0].set_title("(c) Defects4J defect density (bugs / KLOC)")
ax[1, 0].legend()

# (d) positive-class rate across all sources, common axis, 20% imbalance threshold
rates = {name: to_binary(df[promise_label_cols[name]]).mean() * 100 for name, df in promise_raw.items()}
rates["Defects4J (bug-level, artefact)"] = 100.0  # every raw D4J record is a confirmed bug -- see Sec 3.3.5
pd.Series(rates).plot(kind="barh", ax=ax[1, 1], color="#C1443C")
ax[1, 1].axvline(20, color="black", linestyle="--", linewidth=1, label="20% imbalance threshold")
ax[1, 1].set_title("(d) Positive-class rate across sources")
ax[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig3_1_distribution.png", dpi=150)
plt.show()

## 5. Defects4J harmonisation — method-level feature extraction (Section 3.3.5, Increment I3)

For each bug: checkout the buggy and fixed versions, diff every shared `.java` file,
map changed line ranges onto their enclosing method via `javalang`, label those methods
`defective = 1`, and label the remaining methods in the same modified file(s) `defective = 0`.
CK/McCabe-style metrics are then computed per method from the AST.

**Documented simplification (per Section 3.10 fallback provision):** full CK metrics
(coupling between objects, depth of inheritance tree, response for class, LCOM) require
class-level bytecode analysis (e.g. via `ckjm`) which needs a full project build. This
pipeline computes the subset of metrics obtainable directly from source-level AST parsing
— lines of code, McCabe cyclomatic complexity, parameter count, nesting depth, and token
count — which is consistent with the file-/method-level fallback described in Section 3.10
if full CK extraction proves infeasible within the project timeline. If full CK metrics are
required, integrate `ckjm` over the compiled buggy/fixed jars as a follow-up increment.

In [ ]:
import javalang

def cyclomatic_complexity(method_node):
    """Counts McCabe decision points within a method AST node."""
    decision_types = (javalang.tree.IfStatement, javalang.tree.ForStatement,
                       javalang.tree.WhileStatement, javalang.tree.DoStatement,
                       javalang.tree.SwitchStatementCase, javalang.tree.CatchClause,
                       javalang.tree.TernaryExpression)
    count = 1
    for _, node in method_node.filter(decision_types):
        count += 1
    for _, node in method_node.filter(javalang.tree.BinaryOperation):
        if node.operator in ("&&", "||"):
            count += 1
    return count


def parse_methods(source_text):
    """Returns a list of dicts, one per method: name, start/end line, node, source slice."""
    lines = source_text.splitlines()
    try:
        tree = javalang.parse.parse(source_text)
    except Exception:
        return []
    methods = []
    for _, node in tree.filter(javalang.tree.MethodDeclaration):
        if node.position is None:
            continue
        start = node.position.line
        # heuristic end-of-method: last statement's line, or start if body is empty
        end = start
        for path, sub in node.filter(javalang.tree.Node):
            if getattr(sub, "position", None):
                end = max(end, sub.position.line)
        methods.append(dict(
            name=node.name, start=start, end=end, node=node,
            loc=max(end - start + 1, 1),
            n_params=len(node.parameters),
            cyclomatic=cyclomatic_complexity(node),
            source="\n".join(lines[start - 1:end]),
        ))
    return methods


def changed_line_numbers(buggy_text, fixed_text):
    """Line numbers (1-indexed, buggy-file coordinates) touched by the fix."""
    sm = difflib.SequenceMatcher(None, buggy_text.splitlines(), fixed_text.splitlines())
    changed = set()
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            changed.update(range(i1 + 1, i2 + 1))
    return changed

In [ ]:
def extract_bug_methods(project, bug_id, buggy_root):
    """Checks out one buggy/fixed pair, returns a list of labelled method records."""
    buggy_dir = os.path.join(buggy_root, f"{project}_{bug_id}_buggy")
    fixed_dir = os.path.join(buggy_root, f"{project}_{bug_id}_fixed")
    if not os.path.exists(buggy_dir):
        d4j("checkout", "-p", project, "-v", f"{bug_id}b", "-w", buggy_dir)
    if not os.path.exists(fixed_dir):
        d4j("checkout", "-p", project, "-v", f"{bug_id}f", "-w", fixed_dir)

    modified_classes = d4j("export", "-p", "classes.modified", cwd=buggy_dir).strip().splitlines()
    records = []
    for cls in modified_classes:
        rel_path = cls.replace(".", "/") + ".java"
        buggy_file = _find_file(buggy_dir, rel_path)
        fixed_file = _find_file(fixed_dir, rel_path)
        if not buggy_file or not fixed_file:
            continue
        buggy_src = open(buggy_file, errors="ignore").read()
        fixed_src = open(fixed_file, errors="ignore").read()
        changed_lines = changed_line_numbers(buggy_src, fixed_src)

        for m in parse_methods(buggy_src):
            is_defective = int(bool(set(range(m["start"], m["end"] + 1)) & changed_lines))
            records.append(dict(
                project=project, bug_id=bug_id, class_name=cls, method=m["name"],
                loc=m["loc"], cyclomatic=m["cyclomatic"], n_params=m["n_params"],
                n_tokens=len(m["source"].split()),
                defective=is_defective, source=m["source"],
            ))
    return records


def _find_file(root, rel_path):
    # rel_path may not match exactly if src root differs (src/main/java vs src/java etc.)
    target = rel_path.replace("/", os.sep)
    for r, _, files in os.walk(root):
        candidate = os.path.join(r, os.path.basename(target))
        if os.path.basename(target) in files and r.endswith(os.path.dirname(target)):
            return os.path.join(r, os.path.basename(target))
    for r, _, files in os.walk(root):
        if os.path.basename(target) in files:
            return os.path.join(r, os.path.basename(target))
    return None

In [ ]:
harmonised_records = []
work_root = os.path.abspath("d4j_work")
os.makedirs(work_root, exist_ok=True)

for _, row in d4j_meta.iterrows():
    try:
        recs = extract_bug_methods(row["project"], row["revision.id.buggy"] if "revision.id.buggy" in row else row.get("bug.id"), work_root)
        harmonised_records.extend(recs)
    except Exception as e:
        print(f"skipped {row['project']} bug: {e}")

d4j_harmonised = pd.DataFrame(harmonised_records)
print(d4j_harmonised.shape)
if len(d4j_harmonised):
    print("positive rate:", d4j_harmonised['defective'].mean() * 100, "%")
d4j_harmonised.head()

> **Note on execution risk (Increment I3):** method-level extraction is the highest-risk
> component of the pipeline (Section 3.10). If checkout or parsing fails for a non-trivial
> share of bugs — for example due to non-standard source directory layouts in older project
> versions — fall back to (a) an existing published Defects4J feature set (e.g. the DeepFL
> release), or (b) file-level rather than method-level granularity, and record which fallback
> was used in Chapter 4 rather than silently substituting it here.

## 6. Persist the harmonised Defects4J views (metric view for Pipeline A, source view for Pipeline B)

In [ ]:
d4j_harmonised.to_csv(f"{OUT_DIR}/defects4j_harmonised_full.csv", index=False)

# Pipeline A (ML): metric-only view, no raw source
d4j_metric_view = d4j_harmonised.drop(columns=["source"], errors="ignore")
d4j_metric_view.to_csv(f"{OUT_DIR}/defects4j_harmonised_metrics.csv", index=False)

# Pipeline B (DL): source + label only
d4j_source_view = d4j_harmonised[["project", "bug_id", "method", "source", "defective"]] if len(d4j_harmonised) else pd.DataFrame()
d4j_source_view.to_csv(f"{OUT_DIR}/defects4j_harmonised_source.csv", index=False)

print("saved:", d4j_metric_view.shape, d4j_source_view.shape)

## 7. Pipeline A — shared ML pre-processing utilities (Sections 3.4.1, 3.4.2)

Correlation pruning -> RFE -> scaling (SVM only) -> SMOTE (skipped for PC2, Section 3.3.4).
Every transform is fitted on the training portion of each fold only.

In [ ]:
class CorrPruner(BaseEstimator, TransformerMixin):
    """Drops one feature from each pair with |corr| above threshold. Fitted on X only."""
    def __init__(self, thresh=0.9):
        self.thresh = thresh

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [c for c in upper.columns if any(upper[c] > self.thresh)]
        self.keep_idx_ = [i for i, c in enumerate(X.columns) if c not in drop]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.iloc[:, self.keep_idx_].values


def select_features(Xtr, ytr, Xval, k=15):
    """Correlation pruning + RFE fitted on the training fold only."""
    pruner = CorrPruner(0.9).fit(Xtr)
    Xtr_p = pd.DataFrame(pruner.transform(Xtr))
    Xval_p = pd.DataFrame(pruner.transform(Xval))
    n_feats = min(k, Xtr_p.shape[1])
    if n_feats < 1:
        return Xtr_p, Xval_p
    rfe = RFE(RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE), n_features_to_select=n_feats)
    rfe.fit(Xtr_p, ytr)
    return Xtr_p.loc[:, rfe.support_], Xval_p.loc[:, rfe.support_]


def make_ml_pipeline(clf, scale=False, use_smote=True):
    steps = []
    if scale:
        steps.append(("scale", StandardScaler()))
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("clf", clf))
    return ImbPipeline(steps)


ML_MODELS = {
    "SVM": dict(
        build=lambda smote: make_ml_pipeline(SVC(probability=True, random_state=RANDOM_STATE), scale=True, use_smote=smote),
        grid={"clf__C": [0.1, 1, 10], "clf__kernel": ["rbf", "linear"], "clf__gamma": ["scale", "auto"]},
        search="grid",
    ),
    "Decision Tree": dict(
        build=lambda smote: make_ml_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE), use_smote=smote),
        grid={"clf__max_depth": [5, 10, 20, None], "clf__min_samples_split": [2, 5, 10],
              "clf__criterion": ["gini", "entropy"]},
        search="grid",
    ),
    "Random Forest": dict(
        build=lambda smote: make_ml_pipeline(RandomForestClassifier(random_state=RANDOM_STATE), use_smote=smote),
        grid={"clf__n_estimators": [100, 200, 400], "clf__max_depth": [None, 10, 20],
              "clf__min_samples_split": [2, 5], "clf__max_features": ["sqrt", "log2"]},
        search="random",
    ),
}

## 8. Unified Pipeline-A CV + evaluation harness (Section 3.6, 3.7)

One function runs stratified 10-fold CV (repeated stratified 5-fold for PC2) with nested
hyper-parameter search for all three ML models on a given feature table, so PROMISE (KC1,
PC1, PC2) and the harmonised Defects4J metric view are evaluated identically.

In [ ]:
def run_pipeline_a(X, y, dataset_name, use_smote=True, n_folds=None, repeats=1):
    n_folds = n_folds or N_OUTER_ML
    if repeats > 1:
        cv = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=repeats, random_state=RANDOM_STATE)
    else:
        cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    fold_metrics, fold_timing = {m: [] for m in ML_MODELS}, {m: [] for m in ML_MODELS}

    for name, spec in ML_MODELS.items():
        for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
            Xtr, Xval = X.iloc[tr_idx].reset_index(drop=True), X.iloc[val_idx].reset_index(drop=True)
            ytr, yval = y[tr_idx], y[val_idx]

            Xtr_sel, Xval_sel = select_features(Xtr, ytr, Xval)
            if Xtr_sel.shape[1] == 0:
                continue

            pipe = spec["build"](use_smote)
            searcher = (GridSearchCV(pipe, spec["grid"], scoring="f1", cv=3, n_jobs=-1)
                        if spec["search"] == "grid" else
                        RandomizedSearchCV(pipe, spec["grid"], scoring="f1", cv=3, n_iter=10,
                                            n_jobs=-1, random_state=RANDOM_STATE))
            t0 = time.time()
            searcher.fit(Xtr_sel, ytr)
            train_time = time.time() - t0

            t0 = time.time()
            pred = searcher.predict(Xval_sel)
            infer_time = (time.time() - t0) / max(len(Xval_sel), 1)
            proba = searcher.predict_proba(Xval_sel)[:, 1]

            fold_metrics[name].append(dict(
                dataset=dataset_name, fold=fold_i,
                accuracy=accuracy_score(yval, pred),
                precision=precision_score(yval, pred, zero_division=0),
                recall=recall_score(yval, pred, zero_division=0),
                f1=f1_score(yval, pred, zero_division=0),
                auc=roc_auc_score(yval, proba) if len(set(yval)) > 1 else np.nan,
                mcc=matthews_corrcoef(yval, pred),
            ))
            fold_timing[name].append(dict(dataset=dataset_name, train_s=train_time, infer_s=infer_time))

    return fold_metrics, fold_timing

## 9. Run Pipeline A across every dataset (Increment I2 + I5)

KC1 / PC1 use standard 10-fold CV with SMOTE. PC2, per Section 3.3.4, is evaluated
separately with repeated stratified 5-fold CV and **no SMOTE** (23 positives is too few
to synthesise credibly). The harmonised Defects4J metric view is evaluated identically
to KC1/PC1 so that it forms the common ground for the six-model comparison (Section 3.3.5).

In [ ]:
all_fold_metrics = {}
all_fold_timing = {}

ID_COLS = {"index", "id"}

def prep_xy(df, label_col):
    feats = [c for c in df.columns if c.lower() not in ID_COLS and c != label_col]
    X = df[feats].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median())
    y = to_binary(df[label_col]).values
    return X, y

for name, df in promise_raw.items():
    X, y = prep_xy(df, promise_label_cols[name])
    if name == "PC2":
        fm, ft = run_pipeline_a(X, y, "PC2", use_smote=False, n_folds=PC2_REPEATS, repeats=PC2_REPEATS)
    else:
        fm, ft = run_pipeline_a(X, y, name, use_smote=True, n_folds=N_OUTER_ML)
    all_fold_metrics[name] = fm
    all_fold_timing[name] = ft
    print(f"done: {name}")

if len(d4j_metric_view):
    Xd, yd = prep_xy(d4j_metric_view.select_dtypes(include=[np.number]).assign(defective=d4j_metric_view["defective"]), "defective")
    fm, ft = run_pipeline_a(Xd, yd, "Defects4J (harmonised)", use_smote=True, n_folds=N_OUTER_ML)
    all_fold_metrics["Defects4J"] = fm
    all_fold_timing["Defects4J"] = ft
    print("done: Defects4J (harmonised, Pipeline A)")

## 10. Pipeline B — deep learning pre-processing (Section 3.4.3, Increment I4)

Defects4J raw method source only (PROMISE provides no source code, Section 3.3.5).
Tokenises each method with `javalang`'s lexer, builds a vocabulary from the training
folds only, and pads/truncates to a fixed length.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

def tokenize_source(source):
    try:
        return [tok.value for tok in javalang.tokenizer.tokenize(source)]
    except Exception:
        return source.split()

def build_vocab(token_lists, min_freq=2):
    from collections import Counter
    counts = Counter(tok for toks in token_lists for tok in toks)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tok, c in counts.items():
        if c >= min_freq:
            vocab[tok] = len(vocab)
    return vocab

def encode(token_lists, vocab, max_len):
    seqs = [[vocab.get(t, vocab["<UNK>"]) for t in toks][:max_len] for toks in token_lists]
    return tf.keras.preprocessing.sequence.pad_sequences(seqs, maxlen=max_len, padding="post")

MAX_LEN = 200  # informed by training-fold token-length distribution -- see below

In [ ]:
if len(d4j_source_view):
    token_lists_all = d4j_source_view["source"].apply(tokenize_source)
    lens = token_lists_all.apply(len)
    print(lens.describe())
    MAX_LEN = int(np.percentile(lens, 90)) or 200
    plt.figure(figsize=(6, 3))
    plt.hist(lens, bins=30)
    plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"90th pct = {MAX_LEN}")
    plt.title("Token-length distribution — Defects4J methods")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig_token_length.png", dpi=150)
    plt.show()

## 11. Pipeline B — CNN, LSTM and hybrid CNN-LSTM model builders (Section 3.5.2, 3.6)

In [ ]:
def build_cnn(vocab_size, max_len, filters=64, kernel_size=5, dropout=0.3, lr=1e-3):
    m = models.Sequential([
        layers.Embedding(vocab_size, 64, input_length=max_len),
        layers.Conv1D(filters, kernel_size, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(dropout),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_lstm(vocab_size, max_len, units=64, dropout=0.3, lr=1e-3):
    m = models.Sequential([
        layers.Embedding(vocab_size, 64, input_length=max_len),
        layers.LSTM(units, dropout=dropout),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn_lstm(vocab_size, max_len, filters=64, kernel_size=5, units=64, dropout=0.3, lr=1e-3):
    m = models.Sequential([
        layers.Embedding(vocab_size, 64, input_length=max_len),
        layers.Conv1D(filters, kernel_size, activation="relu"),
        layers.MaxPooling1D(2),
        layers.LSTM(units, dropout=dropout),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

DL_MODELS = {
    "CNN": build_cnn,
    "LSTM": build_lstm,
    "Hybrid CNN-LSTM": build_cnn_lstm,
}

## 12. Pipeline B CV + evaluation harness (Increment I4/I5)

Trains and evaluates the three DL models on the harmonised Defects4J token sequences
using stratified k-fold CV, class-weighted loss (no SMOTE — Section 3.4.3) and early
stopping on validation loss.

In [ ]:
def run_pipeline_b(source_view, n_folds=None, epochs=None):
    n_folds = n_folds or N_OUTER_DL
    epochs = epochs or DL_EPOCHS

    y_all = source_view["defective"].values
    tokens_all = source_view["source"].apply(tokenize_source).tolist()

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_metrics = {m: [] for m in DL_MODELS}
    fold_timing = {m: [] for m in DL_MODELS}

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(np.zeros(len(y_all)), y_all)):
        train_tokens = [tokens_all[i] for i in tr_idx]
        val_tokens = [tokens_all[i] for i in val_idx]
        ytr, yval = y_all[tr_idx], y_all[val_idx]

        vocab = build_vocab(train_tokens)          # fitted on training fold only
        Xtr = encode(train_tokens, vocab, MAX_LEN)
        Xval = encode(val_tokens, vocab, MAX_LEN)

        cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=ytr)
        class_weight = {0: cw[0], 1: cw[1]}

        for name, builder in DL_MODELS.items():
            model = builder(len(vocab), MAX_LEN)
            es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

            t0 = time.time()
            model.fit(Xtr, ytr, validation_split=0.15, epochs=epochs, batch_size=32,
                      class_weight=class_weight, callbacks=[es], verbose=0)
            train_time = time.time() - t0

            t0 = time.time()
            proba = model.predict(Xval, verbose=0).ravel()
            infer_time = (time.time() - t0) / max(len(Xval), 1)
            pred = (proba >= 0.5).astype(int)

            fold_metrics[name].append(dict(
                dataset="Defects4J", fold=fold_i,
                accuracy=accuracy_score(yval, pred),
                precision=precision_score(yval, pred, zero_division=0),
                recall=recall_score(yval, pred, zero_division=0),
                f1=f1_score(yval, pred, zero_division=0),
                auc=roc_auc_score(yval, proba) if len(set(yval)) > 1 else np.nan,
                mcc=matthews_corrcoef(yval, pred),
            ))
            fold_timing[name].append(dict(dataset="Defects4J", train_s=train_time, infer_s=infer_time))

    return fold_metrics, fold_timing

In [ ]:
if len(d4j_source_view) >= 20:  # need enough rows for CV to be meaningful
    dl_fold_metrics, dl_fold_timing = run_pipeline_b(d4j_source_view, n_folds=N_OUTER_DL, epochs=DL_EPOCHS)
    all_fold_metrics["Defects4J_DL"] = dl_fold_metrics
    all_fold_timing["Defects4J_DL"] = dl_fold_timing
    print("done: Defects4J (Pipeline B, DL)")
else:
    print("Skipped Pipeline B: too few harmonised Defects4J methods extracted for CV. "
          "Increase BUGS_PER_PROJECT / set QUICK_TEST=False and re-run Sections 3, 5.")

## 13. Consolidated results table (Section 3.7)

Every model/dataset combination reported with the five headline metrics plus MCC.
Accuracy is reported for completeness but, per Section 3.3.4/3.7, **F1 and AUC are the
primary decision metrics** — this is especially important for PC2 and for the harmonised
Defects4J view, both of which are severely imbalanced.

In [ ]:
def summarise(fold_metrics_dict):
    rows = []
    for dataset_key, model_dict in fold_metrics_dict.items():
        for model_name, records in model_dict.items():
            if not records:
                continue
            df = pd.DataFrame(records)
            agg = df[["accuracy", "precision", "recall", "f1", "auc", "mcc"]].mean()
            rows.append(dict(dataset=dataset_key, model=model_name, **agg.to_dict(),
                              n_folds=len(df)))
    return pd.DataFrame(rows)

results_summary = summarise(all_fold_metrics)
results_summary = results_summary.sort_values(["dataset", "f1"], ascending=[True, False])
results_summary.to_csv(f"{OUT_DIR}/results_summary.csv", index=False)
results_summary

In [ ]:
def summarise_timing(fold_timing_dict):
    rows = []
    for dataset_key, model_dict in fold_timing_dict.items():
        for model_name, records in model_dict.items():
            if not records:
                continue
            df = pd.DataFrame(records)
            rows.append(dict(dataset=dataset_key, model=model_name,
                              mean_train_s=df["train_s"].mean(),
                              mean_infer_s=df["infer_s"].mean()))
    return pd.DataFrame(rows)

timing_summary = summarise_timing(all_fold_timing)
timing_summary.to_csv(f"{OUT_DIR}/timing_summary.csv", index=False)
timing_summary

## 14. Visual comparison — performance and computational cost (RQ1, RQ2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

pivot_f1 = results_summary.pivot(index="model", columns="dataset", values="f1")
pivot_f1.plot(kind="bar", ax=ax[0])
ax[0].set_title("Mean F1-score by model and dataset")
ax[0].set_ylabel("F1")
ax[0].legend(fontsize=7)

pivot_time = timing_summary.pivot(index="model", columns="dataset", values="mean_train_s")
pivot_time.plot(kind="bar", ax=ax[1], logy=True)
ax[1].set_title("Mean training time per fold (log scale, s)")
ax[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_model_comparison.png", dpi=150)
plt.show()

In [ ]:
# Accuracy-vs-F1 divergence check -- makes the Section 3.7 PC2 caveat visible in the results
if "PC2" in results_summary["dataset"].values:
    pc2 = results_summary[results_summary["dataset"] == "PC2"][["model", "accuracy", "f1", "auc"]]
    print("PC2 — accuracy is inflated by class imbalance; F1/AUC are the metrics that matter:")
    print(pc2.to_string(index=False))

## 15. Statistical significance testing (Section 3.8)

Wilcoxon signed-rank test on paired per-fold F1 scores, Holm-Bonferroni corrected across
the 15 pairwise comparisons among six models. Two separate comparisons are run, following
the constraint identified in Section 3.3.5/3.11:

- **Full six-model comparison** on the harmonised Defects4J view, where ML and DL models
  share identical folds (the only fully-crossed comparison).
- **ML-only comparison** across KC1 / PC1 / PC2 to address RQ3 (consistency across datasets),
  without claiming DL vs ML superiority from a different data source.

In [ ]:
from itertools import combinations

def significance_table(fold_metrics_dict, dataset_keys, metric="f1"):
    frames = {}
    for dk in dataset_keys:
        for model_name, records in fold_metrics_dict.get(dk, {}).items():
            if records:
                frames[model_name] = pd.DataFrame(records)[metric].values

    pairs = list(combinations(frames.keys(), 2))
    rows = []
    raw_p = []
    for a, b in pairs:
        fa, fb = frames[a], frames[b]
        n = min(len(fa), len(fb))
        if n < 2:
            continue
        stat, p = wilcoxon(fa[:n], fb[:n])
        diffs = fa[:n] - fb[:n]
        r = 1 - (2 * stat) / (n * (n + 1) / 2) if n > 0 else np.nan
        rows.append(dict(pair=f"{a} vs {b}", wilcoxon_stat=stat, raw_p=p, rank_biserial_effect=r))
        raw_p.append(p)

    if not rows:
        return pd.DataFrame()
    reject, corrected_p, _, _ = multipletests(raw_p, alpha=0.05, method="holm")
    out = pd.DataFrame(rows)
    out["holm_corrected_p"] = corrected_p
    out["significant (a=0.05)"] = reject
    return out

In [ ]:
# Full six-model comparison on the harmonised Defects4J view (Pipeline A + Pipeline B results)
d4j_combined_metrics = {"Defects4J_all": {}}
d4j_combined_metrics["Defects4J_all"].update(all_fold_metrics.get("Defects4J", {}))
d4j_combined_metrics["Defects4J_all"].update(all_fold_metrics.get("Defects4J_DL", {}))

sig_six_model = significance_table(d4j_combined_metrics, ["Defects4J_all"], metric="f1")
sig_six_model.to_csv(f"{OUT_DIR}/significance_six_model_defects4j.csv", index=False)
sig_six_model

In [ ]:
# ML-only comparison across PROMISE datasets (RQ3 -- consistency across characteristics)
for dataset_key in ["KC1", "PC1", "PC2"]:
    if dataset_key in all_fold_metrics:
        print(f"--- {dataset_key} ---")
        sig = significance_table(all_fold_metrics, [dataset_key], metric="f1")
        sig.to_csv(f"{OUT_DIR}/significance_{dataset_key}.csv", index=False)
        print(sig.to_string(index=False) if len(sig) else "not enough paired folds")

## 16. Cross-dataset consistency analysis (RQ3)

Plots each ML model's F1-score across all four ML-evaluated sources (KC1, PC1, PC2,
Defects4J-harmonised) to visualise how consistently each model performs as class balance
and dataset size vary (Section 3.3.4).

In [ ]:
ml_only = results_summary[results_summary["model"].isin(ML_MODELS.keys())]
pivot_consistency = ml_only.pivot(index="dataset", columns="model", values="f1")

pivot_consistency.plot(kind="line", marker="o", figsize=(8, 5))
plt.title("Model consistency across datasets of differing imbalance (RQ3)")
plt.ylabel("F1-score")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_rq3_consistency.png", dpi=150)
plt.show()

## 17. Save all artefacts

All figures, per-fold metrics, summary tables and significance tests are written to
`outputs/` for direct inclusion in Chapter 4.

In [ ]:
with open(f"{OUT_DIR}/fold_metrics_raw.json", "w") as f:
    json.dump({k: {m: v for m, v in d.items()} for k, d in all_fold_metrics.items()}, f, default=str, indent=2)

print("Artefacts written to:", os.path.abspath(OUT_DIR))
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

## 18. Chapter summary and next steps

This notebook implements every phase of Chapter 3's methodology end to end:

- **Section 3.3** — acquisition and distribution analysis of PROMISE (KC1/PC1/PC2) and Defects4J.
- **Section 3.3.5** — harmonisation of Defects4J into a method-level metric view (Pipeline A) and a raw-source view (Pipeline B).
- **Section 3.4** — the two divergent pre-processing pipelines, converging on one evaluation protocol.
- **Section 3.5/3.6** — all six models, hyper-parameter search, stratified k-fold CV (10-fold, 5-fold-repeated for PC2).
- **Section 3.7** — five metrics + MCC, with F1/AUC treated as primary given class imbalance.
- **Section 3.8** — Wilcoxon signed-rank + Holm-Bonferroni testing, both for the fully-crossed six-model Defects4J comparison and the ML-only PROMISE comparison.

**Before treating results as final:**
1. Re-run with `QUICK_TEST = False` on a GPU runtime for the full 10-fold protocol and the complete Defects4J bug set (Table 3.1).
2. Confirm the harmonised Defects4J positive rate against the ~1% projection in Table 3.3 (Section 3.3.5) and revisit the imbalance strategy if it differs materially.
3. If Increment I3 (method-level extraction) proves unreliable at scale, apply the documented fallback (Section 3.10) and record that decision in Chapter 4.
4. Run the optional cross-project evaluation and GitHub supplementary-source extension (Sections 3.2, 3.3.3) if time permits.